# Decoding Customer Value

## Imports

In [ ]:
import pandas as pd
import numpy as np

## Load & Inspect Data

In [ ]:
cd = pd.read_csv('Dataset.csv')  # customer data
cd.head()
cd.info()
cd.isnull().sum()
cd = cd.drop_duplicates()

## Standardising Columns

In [ ]:
cd.columns = cd.columns.str.strip()
clmn = ['Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season',
        'Subscription Status', 'Shipping Type', 'Discount Applied', 'Promo Code Used',
        'Payment Method', 'Frequency of Purchases']

for i in clmn:
    cd[i] = cd[i].str.title()

## Impute Missing Review Rating (new columns 1-2)

In [ ]:
cd["Rating_Imputed"] = cd["Review Rating"].isnull().astype(int)
median_rating = cd["Review Rating"].median()
cd["Review Rating"] = cd["Review Rating"].fillna(median_rating)

## Convert Yes/No Columns to Binary (new columns 3-4-5)

In [ ]:
cd["Discount Applied"] = cd['Discount Applied'].map({'Yes': 1, 'No': 0})

cd['Promo Used'] = cd['Promo Code Used'].map({'Yes': 1, 'No': 0})

cd['Subscription Used'] = cd['Subscription Status'].map({'Yes': 1, 'No': 0})

## Section 1 — Foundation Features

In [ ]:
# F-01 (new column 6)
freq_map = {'Fortnightly': 26, 'Weekly': 52, 'Annually': 1, 'Bi-Weekly': 26,
             'Monthly': 12, 'Every 3 Months': 4, 'Quarterly': 4}
cd['Freq_per_Year'] = cd['Frequency of Purchases'].map(freq_map)

## Section 2 — Customer Value Features (new columns 7-8-9)

In [ ]:
# F-02
cd["Est_Annual_Spend"] = cd["Purchase Amount (USD)"] * cd["Freq_per_Year"]
# F-03
cd["CLV_Score"] = cd["Est_Annual_Spend"] * np.log1p(cd["Previous Purchases"])
# F-04
cd["Value_Tier"] = pd.qcut(cd["CLV_Score"], q=4, labels=["Low", "Mid", "High", "Premium"])

## Section 3 — Promotional Behaviour Features (new columns 10-11)

In [ ]:
# F-05
cd["Promo_Dependent"] = ((cd["Discount Applied"] == 1) & (cd["Promo Used"] == 1)).astype(int)
# F-06
def satisfaction_fn(rating):
    if rating < 3.0:
        return "Detractor"
    elif rating < 4.0:
        return "Neutral"
    elif rating >= 4.0:
        return "Promoter"
    else:
        pass

cd["Satisfaction_Flag"] = cd["Review Rating"].apply(satisfaction_fn)

## Section 4 — Demographic Segmentation

In [ ]:
# F-07
cd["Age_Group"] = pd.cut(
    cd["Age"],
    bins=[17, 25, 35, 45, 55, 70],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)

## Section 5 — Two Competing Loyalty Definitions

In [ ]:
def min_max_norm(series):  # min-max normalise a series to [0, 1]
    mn, mx = series.min(), series.max()
    if mx > mn:
        ans = (series - mn) / (mx - mn)
    else:
        ans = series * 0.0
    return ans

# normalising
Freq_Norm = min_max_norm(cd['Freq_per_Year'])
Tenure_Norm = min_max_norm(cd['Previous Purchases'])
CLV_Norm = min_max_norm(cd["CLV_Score"])
Rating_Norm = min_max_norm(cd["Review Rating"])
Promo_Ind = 1 - cd["Promo_Dependent"]  # promo-independent

### Loyalty Score A & B

In [ ]:
cd['Loyalty_Score_A'] = (0.40 * Freq_Norm + 0.40 * Tenure_Norm + 0.20 * Promo_Ind)

threshold_A = cd['Loyalty_Score_A'].quantile(0.70)
cd['Loyal_A'] = (cd['Loyalty_Score_A'] >= threshold_A).astype(int)

cd['Loyalty_Score_B'] = (0.50 * CLV_Norm + 0.30 * Rating_Norm + 0.20 * Promo_Ind)

threshold_B = cd['Loyalty_Score_B'].quantile(0.70)
cd['Loyal_B'] = (cd['Loyalty_Score_B'] >= threshold_B).astype(int)

### Loyal in Both

In [ ]:
cd['Loyal_Both'] = ((cd['Loyal_A'] == 1) & (cd['Loyal_B'] == 1)).astype(int)

## Section 6 — Customer Segment

In [ ]:
# F-08
def assign_segment(row):
    vt = row['Value_Tier']
    prdp = row['Promo_Dependent']
    pp = row['Previous Purchases']

    if vt == 'Premium' and prdp == 0:
        return 'Champions'
    elif vt in ('Premium', 'High') and prdp == 1:
        return 'Promo Reliant'
    elif vt in ('Low', 'Mid') and pp >= 30:
        return 'At-Risk Loyalists'
    elif vt in ('Low', 'Mid') and prdp == 1:
        return 'Bargain Hunters'
    else:
        return 'Growth Prospects'

cd['Segment'] = cd.apply(assign_segment, axis=1)

## Export Results

In [ ]:
cd.to_csv('customers_phase1.csv', index=False)